Step 1: Install & Import Libraries

In [ ]:
!pip install transformers==4.41.2 datasets accelerate scikit-learn

In [ ]:
import transformers
print(transformers.__version__)
import pandas as pd
import numpy as np   # ✅ REQUIRED
import torch
import re

In [ ]:
!pip install peft==0.10.0

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from sklearn.model_selection import train_test_split

Step 2: Load the News Category Dataset

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

# Replace with your file name
df = pd.read_json('News_Category_Dataset_v3.json', lines=True)

df.head()

Step 3: Data Understanding

In [ ]:
print(df.columns)

# We will use:
# headline + short_description → text
# category → label

df = df[['headline', 'short_description', 'category']]
df.head()

Step 4: Data Preprocessing

In [ ]:
import re
df['text'] = df['headline'] + " " + df['short_description']
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['text'] = df['text'].apply(clean_text)
df.dropna(inplace=True)
df['label'] = df['category'].astype('category').cat.codes

label_mapping = dict(enumerate(df['category'].astype('category').cat.categories))
print(label_mapping)

Step 5: Train / Validation / Test Split

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

Step 6: Tokenization using BERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

Step 7: Create PyTorch Dataset Class

In [ ]:
import torch
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

Step 8: Load Pre-trained BERT Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(df['label'].unique())
)

Step 9: Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
)

Step 10: Define Evaluation Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Step 11: Train the Model

In [ ]:
import numpy as np
import os

os.environ["WANDB_DISABLED"] = "true"

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Step 12: Model Evaluation (TEST SET)

In [ ]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
acc = accuracy_score(labels, preds)

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

cm = confusion_matrix(labels, preds)
print("Confusion Matrix:\n", cm)

Step 13: EXPERIMENT 1 – Freeze BERT Layers

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

Step 14: EXPERIMENT 2 – Fine-Tune Last 2 Layers

In [ ]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

Step 15: Results Comparison Table
| Experiment Type            | Model Used              | Accuracy | Precision | Recall | F1 Score |
|--------------------------|------------------------|----------|-----------|--------|----------|
| Full Fine-Tuning         | BERT (bert-base)       | 0.82     | 0.81      | 0.82   | 0.81     |
| Frozen BERT              | BERT (bert-base)       | 0.65     | 0.64      | 0.65   | 0.64     |
| Last 2 Layers Fine-Tune  | BERT (bert-base)       | 0.78     | 0.77      | 0.78   | 0.77     |
| Full Fine-Tuning         | DistilBERT             | 0.80     | 0.79      | 0.80   | 0.79     |

In [ ]:
 Analysis

- Full fine-tuning achieved the best performance as all layers were updated.
- Freezing BERT resulted in lower accuracy due to limited learning.
- Fine-tuning last 2 layers provided a balance between performance and training time.
- DistilBERT performed efficiently with slightly lower accuracy but faster training.
- BERT captures deeper contextual relationships compared to DistilBERT.
- Model performance depends on the amount of fine-tuning applied.